### **Setup: Imports und Initialisierung**

In [ ]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('..'))

from quinex_application.postprocessing.regex_mapping import RegexMapping
from quinex_application.visualisation.quinex_visualisation import QuinexVisualisation
from quinex_application.quinex_processing.quinex_postprocessing import QuinexPostprocessing

In [ ]:
discussion = RegexMapping("../data/visualisation_files/regex/entity_patterns.json", "../data/visualisation_files/regex/property_patterns.json")
visualisation = QuinexVisualisation()
postprocessing = QuinexPostprocessing()

### **Load Results**

In [ ]:
df_quinex_metadata = discussion.load_df("../data/quinex/df_quinex_metadata.parquet")
df_quinex = discussion.load_df("../data/quinex/df_quinex.parquet")

In [ ]:
df_sample = df_quinex[(df_quinex["entity"] != "") & (df_quinex["property"] != "")].sample(n=500)
df_sample = df_sample.applymap(
    lambda x: f"'{x}" if isinstance(x, str) and x.startswith('=') else x
)
df_sample.to_csv(
    "technoeconomic_data_500.csv",
    index=False,
    sep=';',
    decimal=',',
)

### **Filter data**

In [ ]:
df_quinex[df_quinex["property"].str.contains("trl", case=False)]["property"].value_counts().head(20)

In [ ]:
df = df_quinex.copy()
entity_regex_map = discussion.entity_regex_map
property_regex_map = discussion.property_regex_map

In [ ]:
# property_regex_map = { 
#     "Cost": {
#         "Capital Cost": property_regex_map["Cost"]["Capital Cost"] 
#     }
# }
# property_regex_map

In [ ]:
# entity_regex_map = {
#     "Wind": {
#         "Onshore Wind": entity_regex_map["Wind"]["Onshore Wind"]
#     }
# }
# entity_regex_map

In [ ]:
df["entity_mapping"] = df["entity"].apply(lambda x: discussion.map_entity(x, entity_regex_map))
label_to_base = discussion.create_label_to_base(entity_regex_map)
df["entity_base_mapping"] = df["entity_mapping"].apply(lambda x: discussion.map_base_from_mapping(x, label_to_base))
df = df[df["entity_mapping"].notna()]

In [ ]:
df["property_mapping"] = df["property"].apply(lambda x: discussion.map_entity(x, property_regex_map))
label_to_base = discussion.create_label_to_base(property_regex_map)
df["property_base_mapping"] = df["property_mapping"].apply(lambda x: discussion.map_base_from_mapping(x, label_to_base))
df = df[df["property_mapping"].notna()]

In [ ]:
df.to_parquet("../data/visualisation_files/df_technologies.parquet")

### **Postprocess Units**

In [ ]:
import pandas as pd
technologies_df = pd.read_parquet("../data/visualisation_files/df_technologies.parquet")
print(technologies_df.columns)
technologies_df = technologies_df[technologies_df["property_mapping"] == "Capital Cost"]
def dicts_to_tuples(units):
    if units is None:
        return None
    return [(u.get('text', ''), u.get('exponent', None), u.get('unit_uri', ''), u.get('year', None)) 
            for u in units]


technologies_df['unit_normalized'] = technologies_df['unit_normalized'].apply(dicts_to_tuples)
print(technologies_df["entity_base_mapping"].value_counts())

In [ ]:
# $/kW Dimension
technologies_df = technologies_df[(technologies_df["unit_dimension"].isin(["A0E0L-2I0M-1H0T3D0", ]))]

# $/kWh Dimension
# technologies_df = technologies_df[(technologies_df["unit_dimension"].isin(["A0E0L-2I0M-1H0T2D0"]))]

# $/kg Dimension
# technologies_df = technologies_df[(technologies_df["unit_dimension"].isin(["A0E0L0I0M-1H0T0D0"]))]

# MW Dimension
# technologies_df = technologies_df[(technologies_df["unit_dimension"].isin(["A0E0L2I0M1H0T-3D0"]))]

In [ ]:
def normalize_compound_units(units):
    """
    Zerlegt zusammengesetzte Units (z.B. €/kW) in ihre Komponenten.
    """
    if units is None:
        return None
    
    result = []
    
    for unit_tuple in units:
        text, exponent, uri, year = unit_tuple
        
        # Prüfe auf EUR-PER-KiloW Muster
        if "EUR-PER-KiloW" in uri:
            result.append(("€", 1.0, "http://qudt.org/vocab/unit/CCY_EUR", year))
            result.append(("kW", -1.0, "http://qudt.org/vocab/unit/KiloW", year))
        # Prüfe auf EUR-PER-W Muster
        elif "EUR-PER-W" in uri:
            result.append(("€", 1.0, "http://qudt.org/vocab/unit/CCY_EUR", year))
            result.append(("W", -1.0, "http://qudt.org/vocab/unit/W", year))
        else:
            # Nicht zusammengesetzt → behalte original
            result.append(unit_tuple)
    
    return result

# Wende die Normalisierung an
technologies_df['unit_normalized'] = technologies_df['unit_normalized'].apply(normalize_compound_units)

In [ ]:
print(technologies_df["unit_normalized"].value_counts())
print("--------------------------------------------------------------------------------")
exclude_values = [
    [(None, None, 'http://qudt.org/vocab/unit/UNITLESS', None)],
    [('%', 1.0, 'http://qudt.org/vocab/unit/PERCENT', None)], 
]
technologies_df = technologies_df[~technologies_df["unit_normalized"].isin(exclude_values)]
print(technologies_df["unit_normalized"].value_counts().head(20))

In [ ]:
# --- 1. Erste Normalisierung: USD pro Watt (oder GW) ---
# target_unit_full = [("$", 1.0, "http://qudt.org/vocab/currency/USD", None),
#                     ("kWh", -1.0, "http://qudt.org/vocab/unit/KiloW-HR", None)]
target_unit_full = [("$", 1.0, "http://qudt.org/vocab/unit/CCY_USD", None),
                    ("W", -1.0, "http://qudt.org/vocab/unit/W", None)]
# target_unit_full = [("$", 1.0, "http://qudt.org/vocab/unit/CCY_USD", None),
#                     ("kg", -1.0, "http://qudt.org/vocab/unit/KiloGM", None)]
# target_unit_full = [("MW", 1.0, "http://qudt.org/vocab/unit/MegaW", None)]

technologies_df['value_converted'] = technologies_df.apply(
    lambda row: visualisation.convert_row(
        row, 
        target_unit_full,
        #2024
      ),
    axis=1
)

In [ ]:
technologies_df.head(1)["temporal_scope"]

In [ ]:
# --- 2. Zweite Normalisierung: nur USD, für Zeilen mit None ---
# target_unit_usd = [("$", 1.0, "http://qudt.org/vocab/currency/USD", None)]

# mask_none = technologies_df['value_converted'].isna()

# technologies_df.loc[mask_none, 'value_converted'] = technologies_df.loc[mask_none].apply(
#     lambda row: visualisation.convert_row(row, target_unit_usd),
#     axis=1
# )

# import math

# def scale_usd_per_watt(value):
#     if value is None or (isinstance(value, float) and math.isnan(value)):
#         return value
    
#     abs_val = abs(value)
    
#     if abs_val < 10:           # einstelliger Bereich → USD/W
#         return value
#     elif abs_val < 1e4:        # ca. 10–10.000 → USD/kW
#         return value / 1e3
#     elif abs_val < 1e7:        # 10.000 – 10.000.000 → USD/MW
#         return value / 1e6
#     else:                      # alles darüber hinaus → NaN
#         return float('nan')

# technologies_df['value_converted'] = technologies_df['value_converted'].apply(scale_usd_per_watt)

In [ ]:
technologies_df = technologies_df[technologies_df["value_converted"].notna()]
technologies_df = technologies_df.reset_index(drop=True)
technologies_df.to_excel("../data/visualisation_files/unit_converted/df_technologies_capex.xlsx")

In [ ]:
technologies_df